# Script1 : Approche Classique (TF-IDF + Similarité Cosinus)

In [1]:
pip install pandas numpy scikit-learn sentence-transformers torch

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import re
import html
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

In [3]:
# ÉTAPE 1 : Chargement des fichiers de données
print("--> Chargement des fichiers de données...")
DATA_DIR = "."          # dossier contenant les CSV (modifier si besoin)

corpus_df  = pd.read_csv(f"{DATA_DIR}/medrxiv2026_corpus.csv")                 # ID, title, abstract, date
queries_df = pd.read_csv(f"{DATA_DIR}/medrxiv2026_reco_queries_train.csv")     # query_id, query
qrels_df   = pd.read_csv(f"{DATA_DIR}/medrxiv2026_reco_qrels_train.csv")       # query_id, ID, relevance

print(f"Corpus chargé : {len(corpus_df)} articles")
print(f"Requêtes chargées : {len(queries_df)} requêtes")
print(f"Jugements de pertinence : {len(qrels_df)} lignes, niveaux {sorted(qrels_df['relevance'].unique())}")

--> Chargement des fichiers de données...
Corpus chargé : 5058 articles
Requêtes chargées : 465 requêtes
Jugements de pertinence : 2667 lignes, niveaux [np.int64(1), np.int64(2)]


In [4]:
# ÉTAPE 2 : Nettoyage et Prétraitement des Textes
def nettoyer_texte(texte):
    """
    Nettoyage pour le NLP :
    - décode les entités HTML (&ge; &le; &gt; ...)
    - convertit en minuscules
    - supprime uniquement les VRAIES balises HTML (pas les « < » des p<0.05)
    - supprime la ponctuation et les caractères spéciaux
    - supprime les espaces superflus
    """
    if not isinstance(texte, str):
        return ""
    texte = html.unescape(texte).lower()
    texte = re.sub(r'</?(?:p|b|i|u|sub|sup|br|em|strong|span|div|ul|ol|li)\b[^>]*>', ' ', texte)
    texte = re.sub(r'[^a-z0-9\s]', ' ', texte)
    texte = re.sub(r'\s+', ' ', texte).strip()
    return texte

print("--> Nettoyage des résumés et des requêtes...")

# Concaténation du titre et du résumé pour capturer l'ensemble des informations
corpus_df['clean_title'] = corpus_df['title'].apply(nettoyer_texte)
corpus_df['clean_abstract'] = corpus_df['abstract'].apply(nettoyer_texte)
corpus_df['full_text'] = corpus_df['clean_title'] + " " + corpus_df['clean_abstract']

# Nettoyage des requêtes
queries_df['clean_query'] = queries_df['query'].apply(nettoyer_texte)

# Doublons : 326 groupes de preprints présents en plusieurs versions. On les GARDE : les qrels marquent
# presque toujours toutes les versions comme pertinentes (131 groupes sur 158), en retirer ferait perdre des résultats.
doc_ids = corpus_df['ID'].values

--> Nettoyage des résumés et des requêtes...


In [5]:
# ÉTAPE 3 : Métriques d'évaluation (DCG, nDCG@10, MRR)
TOP_K = 10

def dcg_at_k(relevances, k=10):
    """DCG avec gain 2^rel - 1 : pénalise les articles pertinents placés trop bas."""
    relevances = np.asarray(relevances, dtype=float)[:k]
    if not relevances.size:
        return 0.
    return np.sum((2**relevances - 1) / np.log2(np.arange(2, relevances.size + 2)))

def ndcg_at_k(relevances, relevances_ideales, k=10):
    """
    nDCG@k : DCG des k résultats renvoyés / DCG IDÉAL.
    L'idéal est calculé sur TOUS les documents pertinents connus de la requête (qrels),
    pas sur les résultats renvoyés : sinon les documents manqués ne sont jamais pénalisés.
    """
    idcg = dcg_at_k(sorted(relevances_ideales, reverse=True), k)
    return dcg_at_k(relevances, k) / idcg if idcg else 0.

def rr_at_k(relevances, k=10):
    """Rang réciproque du premier document pertinent dans le top k (0 s'il n'y en a pas)."""
    for i, r in enumerate(relevances[:k]):
        if r > 0:
            return 1.0 / (i + 1)
    return 0.

# Qrels : {query_id: {doc_id: relevance}}
qrels_dict = {}
for q_id, d_id, rel in qrels_df[['query_id', 'ID', 'relevance']].itertuples(index=False):
    qrels_dict.setdefault(q_id, {})[d_id] = rel

def evaluer(S, positions):
    """S : matrice de scores (nb_requêtes x nb_documents). positions : lignes de queries_df à évaluer."""
    nd, rr = [], []
    for i in positions:
        q_id = queries_df.loc[i, 'query_id']
        top = np.argsort(-S[i], kind="stable")[:TOP_K]
        rels = [qrels_dict[q_id].get(d, 0) for d in doc_ids[top]]
        nd.append(ndcg_at_k(rels, list(qrels_dict[q_id].values()), TOP_K))
        rr.append(rr_at_k(rels, TOP_K))
    return {"nDCG@10": round(float(np.mean(nd)), 4), "MRR": round(float(np.mean(rr)), 4)}

# Découpage des REQUÊTES (le corpus est commun au train et au test) : 80 % réglage / 20 % validation
rng = np.random.RandomState(42)
perm = rng.permutation(len(queries_df))
n_dev = int(0.2 * len(perm))
DEV, TRAIN = np.sort(perm[:n_dev]), np.sort(perm[n_dev:])
print(f"{len(TRAIN)} requêtes pour régler, {len(DEV)} pour valider")

372 requêtes pour régler, 93 pour valider


In [6]:
# ÉTAPE 4 : Baseline TF-IDF + similarité cosinus
print(" Vectorisation du corpus avec TF-IDF...")

# - stop_words='english' : filtre les mots vides anglais
# - ngram_range=(1, 1)   : unigrammes. Les bigrammes dégradaient le nDCG@10 sur ce corpus (0,40 -> 0,46 sans eux)
# - pas de max_features  : max_features garde les termes les PLUS FRÉQUENTS (pas les plus informatifs) et coupait les termes rares discriminants
# - sublinear_tf=True    : échelle logarithmique pour réduire l'impact des répétitions
vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 1), sublinear_tf=True)
corpus_vectors = vectorizer.fit_transform(corpus_df['full_text'])
query_vectors = vectorizer.transform(queries_df['clean_query'])
print(f"Dimension de la matrice TF-IDF : {corpus_vectors.shape}")

# Vecteurs L2-normalisés : le produit scalaire est la similarité cosinus (un seul produit matriciel pour toutes les requêtes)
S_tfidf = (query_vectors @ corpus_vectors.T).toarray()

res_tfidf_train = evaluer(S_tfidf, TRAIN)
res_tfidf_dev = evaluer(S_tfidf, DEV)
print("TF-IDF  réglage :", res_tfidf_train)
print("TF-IDF  validation :", res_tfidf_dev)

 Vectorisation du corpus avec TF-IDF...
Dimension de la matrice TF-IDF : (5058, 43326)
TF-IDF  réglage : {'nDCG@10': 0.4656, 'MRR': 0.6481}
TF-IDF  validation : {'nDCG@10': 0.4244, 'MRR': 0.6433}


In [7]:
# ÉTAPE 5 : BM25 (ajout)
class BM25:
    def __init__(self):
        self.cv = CountVectorizer(stop_words='english')
    def fit(self, docs):
        tf = self.cv.fit_transform(docs).tocsr().astype(np.float32)
        N = tf.shape[0]
        df = np.bincount(tf.indices, minlength=tf.shape[1])
        self.idf = np.log(1 + (N - df + 0.5) / (df + 0.5)).astype(np.float32)
        self.tf = tf.tocoo()
        self.dl = np.asarray(tf.sum(1)).ravel()
        self.avgdl = self.dl.mean()
        return self
    def set_params(self, k1=1.2, b=0.75):
        t = self.tf
        denom = t.data + k1 * (1 - b + b * self.dl[t.row] / self.avgdl)
        data = t.data * (k1 + 1) / denom * self.idf[t.col]
        self.W = sp.csr_matrix((data, (t.row, t.col)), shape=t.shape)
        return self
    def scores(self, queries_clean):
        Q = self.cv.transform(queries_clean).tocsr()
        Q.data[:] = 1.0                      # termes de requête binaires
        return (Q @ self.W.T).toarray()

bm25 = BM25().fit(corpus_df['full_text'])
lignes = []
for k1 in [0.6, 0.9, 1.2, 1.5, 2.0]:
    for b in [0.4, 0.6, 0.75, 0.9]:
        bm25.set_params(k1, b)
        lignes.append({"k1": k1, "b": b, **evaluer(bm25.scores(queries_df['clean_query']), TRAIN)})
grille_bm25 = pd.DataFrame(lignes).sort_values("nDCG@10", ascending=False)
print(grille_bm25.head(5).to_string(index=False))
BM_K1, BM_B = grille_bm25.iloc[0][["k1", "b"]]
bm25.set_params(BM_K1, BM_B)
S_bm25 = bm25.scores(queries_df['clean_query'])
print(f"BM25 (k1={BM_K1}, b={BM_B}) validation :", evaluer(S_bm25, DEV))

 k1    b  nDCG@10    MRR
1.2 0.90   0.4999 0.7006
0.9 0.90   0.4999 0.6959
1.5 0.75   0.4984 0.6948
1.2 0.75   0.4977 0.6986
0.9 0.75   0.4975 0.6949
BM25 (k1=1.2, b=0.9) validation : {'nDCG@10': 0.4637, 'MRR': 0.7013}


In [8]:
# ÉTAPE 6 : Expansion de requête par pseudo-relevance feedback (Rocchio) (ajout)
# La requête est enrichie par le centroïde TF-IDF de ses k premiers résultats BM25 : le vocabulaire des abstracts
# les plus proches compense l'écart lexical entre la question et le texte.
def rocchio(S_premier_tour, Q, X, k=5, beta=1.0):
    nq = S_premier_tour.shape[0]
    top = np.argpartition(-S_premier_tour, k - 1, axis=1)[:, :k]
    sel = sp.csr_matrix((np.full(nq * k, 1.0 / k), (np.repeat(np.arange(nq), k), top.ravel())), shape=(nq, X.shape[0]))
    Q2 = normalize(Q + beta * (sel @ X))
    return (Q2 @ X.T).toarray()

lignes = []
for k in [3, 5, 10]:
    for beta in [0.5, 1.0, 2.0]:
        S = rocchio(S_bm25, query_vectors, corpus_vectors, k, beta)
        lignes.append({"k": k, "beta": beta, **evaluer(S, TRAIN)})
grille_prf = pd.DataFrame(lignes).sort_values("nDCG@10", ascending=False)
print(grille_prf.head(5).to_string(index=False))
PRF_K, PRF_BETA = int(grille_prf.iloc[0]["k"]), float(grille_prf.iloc[0]["beta"])
S_prf = rocchio(S_bm25, query_vectors, corpus_vectors, PRF_K, PRF_BETA)
print(f"BM25 + Rocchio (k={PRF_K}, beta={PRF_BETA}) validation :", evaluer(S_prf, DEV))

 k  beta  nDCG@10    MRR
 3   1.0   0.4988 0.6728
 3   0.5   0.4959 0.6710
 5   0.5   0.4945 0.6730
 3   2.0   0.4893 0.6614
 5   1.0   0.4885 0.6578
BM25 + Rocchio (k=3, beta=1.0) validation : {'nDCG@10': 0.4692, 'MRR': 0.6874}


In [9]:
# ÉTAPE 7 : Comparaison sur les requêtes de validation
comparaison = pd.DataFrame({
    "TF-IDF cosinus": evaluer(S_tfidf, DEV),
    "BM25": evaluer(S_bm25, DEV),
    "BM25 + Rocchio": evaluer(S_prf, DEV),
}).T
print(comparaison)
METHODE_FINALE = comparaison["nDCG@10"].idxmax()
print("\nMéthode retenue (meilleur nDCG@10 en validation) :", METHODE_FINALE)

                nDCG@10     MRR
TF-IDF cosinus   0.4244  0.6433
BM25             0.4637  0.7013
BM25 + Rocchio   0.4692  0.6874

Méthode retenue (meilleur nDCG@10 en validation) : BM25 + Rocchio


In [10]:
# ÉTAPE 8 : Génération du fichier de prédiction sur le jeu de test
def scores_requetes(requetes_clean):
    q_vec = vectorizer.transform(requetes_clean)
    if METHODE_FINALE == "TF-IDF cosinus":
        return (q_vec @ corpus_vectors.T).toarray()
    s_bm = bm25.scores(requetes_clean)
    if METHODE_FINALE == "BM25":
        return s_bm
    return rocchio(s_bm, q_vec, corpus_vectors, PRF_K, PRF_BETA)

def generer_predictions_test(fichier_test_queries, fichier_sortie="Groupe5_Projet3_Script1.csv"):
    test_df = pd.read_csv(fichier_test_queries)
    test_df['clean_query'] = test_df['query'].apply(nettoyer_texte)
    S = scores_requetes(test_df['clean_query'])
    resultats = []
    for i, q_id in enumerate(test_df['query_id']):
        top10 = np.argsort(-S[i], kind="stable")[:TOP_K]
        for rang, doc_idx in enumerate(top10, start=1):
            resultats.append({'query_id': q_id, 'ID': doc_ids[doc_idx], 'rank': rang})
    df = pd.DataFrame(resultats)
    assert list(df.columns) == ['query_id', 'ID', 'rank']
    assert set(df['query_id']) == set(test_df['query_id'])
    assert (df.groupby('query_id').size() == TOP_K).all()
    assert not df.duplicated(['query_id', 'ID']).any()
    df.to_csv(fichier_sortie, index=False)
    print(f"Fichier généré : {fichier_sortie} ({len(df)} lignes, {df['query_id'].nunique()} requêtes)")
    return df

predictions_script1 = generer_predictions_test(f"{DATA_DIR}/medrxiv2026_reco_queries_test.csv")

Fichier généré : Groupe5_Projet3_Script1.csv (2000 lignes, 200 requêtes)
